# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/6(화) 오후 · AI 에이전트 · 도구 호출 — 실습</mark>

오전에 `llm_client.py` 를 만들었습니다. 질문을 보내고 답을 JSON 으로 받아 읽을 수 있습니다.

오후에는 LLM 에게 **무엇을 할지 고르게** 합니다. 도구를 만들고 → 고르게 하고 → 우리 코드가 실행하고 → 위험한 일 앞에는 사람을 둡니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 이 파일을 놓는 자리

1. 단톡방에서 받은 이 파일을 `security-agent-toolkit` 폴더 안의 **`agent_core` 폴더**에 넣습니다. 오전 노트북과 같은 폴더입니다.
2. VS Code 에서 `security-agent-toolkit` 폴더를 열고, 왼쪽 목록의 `agent_core` 에서 이 파일을 엽니다.
3. 오른쪽 위 **커널 선택**을 눌러 Python 을 고릅니다.

### 0.2 오늘 오후의 순서

| 교시 | 무엇 |
|---|---|
| 5교시 | AI 에이전트와 도구 호출 — 도구를 만들고 LLM 이 고르게 한다 |
| 6교시 | 라우터 — 이름으로 함수를 찾아 실행한다 · `tool_router.py` |
| 7교시 | 승인 게이트 — 위험한 도구 앞에 사람을 둔다 · `agent_result.json` |
| 8교시 | 오늘 만든 것 확인 · 남은 도전 |

### 0.3 막혔을 때

1. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.
3. 도전 문제는 안 풀고 넘어가도 됩니다.

### 0.4 문제를 푸는 법

1. 코드 셀의 번호 주석이 **무엇을 쓸지** 알려 줍니다. 주석 바로 아래 빈 줄에 코드를 씁니다.
2. 주석에 적힌 **변수 이름을 그대로** 씁니다. 아래 `print` 줄이 그 이름을 쓰기 때문입니다.
3. 미리 채워 둔 줄은 앞에서 배운 것입니다. 고치지 않습니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · AI 에이전트와 도구 호출</mark>

아래 준비 셀 넷을 먼저 실행합니다. **오전을 못 끝냈어도 여기서부터 시작할 수 있습니다.**

1. `requests` 설치
2. 9/29 에 만든 로그 파일 `raw_logs.txt` 다시 만들기
3. `normalized_logs.json` 다시 만들기 — 오후의 도구가 읽는 파일입니다
4. 오전 산출물 `llm_client.py` 다시 만들기 — 오전에 완성했다면 같은 내용으로 덮어씁니다

In [ ]:
%pip install -q requests

In [ ]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34

In [ ]:
import re
import json

PATTERN = r"(?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())
        else:
            unmatched.append(line.strip())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"정규화 {len(rows)}건 / 안 맞음 {len(unmatched)}건")

In [ ]:
%%writefile llm_client.py
import json
import requests

URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"


def read_api_key():
    api_key = None
    with open("../.env", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("=", 1)
            if parts[0] == "GEMINI_API_KEY":
                api_key = parts[1]
    return api_key


def call_llm(question):
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}
    response = requests.post(URL, headers=headers, json=body, timeout=30)
    if response.status_code != 200:
        return ""
    data = response.json()
    return data["choices"][0]["message"]["content"]


def parse_llm_json(text):
    clean = text.replace("```json", "")
    clean = clean.replace("```", "")
    clean = clean.strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        return None

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **AI 에이전트 (agent)** | 묻고 답하는 챗봇과 무엇이 다른가 |
| **도구 호출 (tool use · function calling)** | LLM 이 우리 컴퓨터의 함수를 직접 실행하는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- AI 에이전트 →
- 도구 호출 →

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5 · 도구를 만들고 LLM 이 고르게 한다</mark>

### 왜 필요한가

1. 오전의 LLM 은 로그를 **설명**하기만 했습니다. 로그 파일을 세거나 IP 를 조회하지는 못합니다.
2. 2교시에 봤듯이 LLM 은 조회하지 않고 지어냅니다. **사실은 우리 코드가 확인**해야 합니다.
3. 그래서 일을 나눕니다. 우리가 **도구(함수)** 를 만들어 두고, LLM 은 **어느 도구를 쓸지 고르기만** 합니다. 실행은 우리 코드가 합니다.

### 이 시간에 나오는 말

| 말 | 뜻 | 5교시에서 보는 모양 |
|---|---|---|
| 에이전트 | 목표를 받아 스스로 순서를 정하고 도구를 쓰는 프로그램 | 목표 → 도구 고르기 → 실행 → 답 |
| 도구 | LLM 이 고를 수 있게 우리가 만들어 둔 함수 | `count_failed_logins(args)` |
| 도구 호출 | 모델이 어떤 함수를 어떤 값으로 부를지 정하는 구조 | `{"tool": "count_failed_logins", "args": {"user": "admin"}}` |
| 도구 목록 | 쓸 수 있는 함수의 이름·설명·인자를 적은 글 | `tools_text` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 도구는 딕셔너리 하나를 받는 함수입니다</mark>

도구는 평범한 파이썬 함수입니다. 다만 인자를 **딕셔너리 하나(`args`)** 로 받게 만듭니다. LLM 이 인자를 JSON 으로 주기 때문입니다.

```python
import json

def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

print(count_failed_logins({"user": "kim.cs"}))          # 1
```

1. `args["user"]` 로 어느 계정을 셀지 받습니다.
2. 9/29 오전의 룰 ① 과 같은 코드입니다. 계정이 같고 등급이 `WARN` 인 기록을 셉니다.
3. 결과는 `return` 으로 돌려줍니다. 화면에 출력하지 않습니다. 부른 쪽이 그 값을 다시 씁니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
def count_warn(args):
    rows = [{"user": "jdoe", "level": "WARN"},
            {"user": "jdoe", "level": "INFO"},
            {"user": "ops.kim", "level": "WARN"}]
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

print(count_warn({"user": "jdoe"}))
```

막히면 바로 위 `5.1 도구는 딕셔너리 하나를 받는 함수입니다` 설명을 다시 봅니다.

In [ ]:
def count_warn(args):
    rows = [{"user": "jdoe", "level": "WARN"},
            {"user": "jdoe", "level": "INFO"},
            {"user": "ops.kim", "level": "WARN"}]
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

print(count_warn({"user": "jdoe"}))

✅ `1` — `jdoe` 의 기록은 둘이지만 `WARN` 은 하나입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
choice = {"tool": "count_failed_logins", "args": {"user": "svc_batch"}}

print(choice["tool"])
print(choice["args"]["user"])
```

막히면 바로 위 `5.1 도구는 딕셔너리 하나를 받는 함수입니다` 설명을 다시 봅니다.

In [ ]:
choice = {"tool": "count_failed_logins", "args": {"user": "svc_batch"}}

print(choice["tool"])
print(choice["args"]["user"])

✅ `count_failed_logins` 그리고 `svc_batch` — LLM 이 고른 도구는 이런 딕셔너리로 옵니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · 로그인 실패를 세는 도구 만들기</font></h3></td></tr></table>

계정 하나의 로그인 실패 횟수를 돌려주는 도구 `count_failed_logins` 를 완성하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `4` |

**💡 힌트**

1. 조건은 둘입니다 — 계정이 `args["user"]` 와 같고, 등급이 `WARN` 입니다. `and` 로 잇습니다.
2. 조건에 맞으면 `count` 에 1 을 더합니다.
3. 반복이 끝난 뒤에 `count` 를 돌려줍니다. `return` 은 `for` 와 같은 깊이에 씁니다.

In [ ]:
import json


def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        # 1. row 의 user 가 args 의 user 와 같고 level 이 "WARN" 이면 (if 한 줄)

            # 2. count 에 1 을 더하세요


    # 3. count 를 return 하세요



print(count_failed_logins({"user": "admin"}))

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.2 도구 목록을 보내면 LLM 이 하나를 고릅니다</mark>

프롬프트에 **도구 목록과 목표**를 함께 넣습니다. LLM 은 어느 도구를 어떤 값으로 부를지를 JSON 으로 답합니다.

```python
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '

text = call_llm(tools_text + "목표: admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.")
```

실제로 받은 답입니다.

```
```json
{"tool": "count_failed_logins", "args": {"user": "admin"}}
```
```

1. LLM 은 **고르기만** 했습니다. 로그 파일을 열지도, 함수를 실행하지도 않았습니다.
2. 고른 결과는 글자입니다. 오전에 만든 `parse_llm_json` 으로 딕셔너리로 바꿉니다.
3. **실행은 우리 파이썬 코드가 합니다.** 그래서 실행할지 말지를 우리가 정할 수 있습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-4 · LLM 에게 도구를 고르게 하기</font></h3></td></tr></table>

도구 목록과 목표를 보내고, LLM 이 고른 결과를 **글자 그대로** 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{"tool": "count_failed_logins", "args": {"user": "kim.cs"}}` 가 든 글자. 코드 블록 표시가 붙을 때도 있습니다 |

**💡 힌트**

1. 질문 글자는 `tools_text`, `"목표: "`, `goal` 을 순서대로 이어 붙입니다.
2. `call_llm` 에 질문 글자를 넣으면 답 글자가 돌아옵니다.
3. 아직 아무 도구도 실행되지 않았습니다. 받은 것은 「무엇을 부를지」 를 적은 글자입니다.

In [ ]:
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. prompt 라는 변수를 만들어 tools_text, "목표: ", goal 을 이어 붙이세요


# 2. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담으세요


# 3. text 를 출력하세요

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · 고른 도구의 이름과 인자 꺼내기</font></h3></td></tr></table>

LLM 의 답을 딕셔너리로 바꿔, **도구 이름**과 **인자**를 한 줄씩 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `count_failed_logins` 그리고 `{'user': 'kim.cs'}` |

**💡 힌트**

1. 질문 만들기와 호출은 문제 5-4 와 같습니다.
2. 글자를 딕셔너리로 바꾸는 것은 오전에 만든 `parse_llm_json` 입니다.
3. 도구 이름은 `tool` 칸에, 인자는 `args` 칸에 있습니다.

In [ ]:
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. prompt 라는 변수를 만들어 tools_text, "목표: ", goal 을 이어 붙이세요


# 2. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담으세요


# 3. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요


# 4. choice 의 tool 칸과 args 칸을 한 줄씩 출력하세요 (두 줄)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-6 · 고른 도구를 우리 코드로 실행하기</font></h3></td></tr></table>

LLM 이 `count_failed_logins` 를 골랐으면 그 함수를 **우리 코드로 실행**하고 결과를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `실행 결과: 1` |

**💡 힌트**

1. 고른 도구 이름은 `choice["tool"]` 에 있습니다. `if` 로 이름을 확인합니다.
2. 함수에 넘길 인자는 `choice["args"]` 입니다. 딕셔너리를 그대로 넘깁니다.
3. `kim.cs` 의 로그인 실패는 로그 파일에 한 번 있습니다.

In [ ]:
import json
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
text = call_llm(tools_text + "목표: " + goal)
choice = parse_llm_json(text)
# ── 여기까지 ──

# 1. choice 의 tool 이 "count_failed_logins" 이면 (if 한 줄)

    # 2. result 라는 변수를 만들어 count_failed_logins 에 choice 의 args 를 넣어 받은 값을 담으세요

    # 3. "실행 결과:" 와 result 를 출력하세요

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.3 목표가 바뀌면 다른 도구를 고릅니다</mark>

같은 도구 목록에 목표만 바꿔 보냈을 때 실제로 받은 답입니다.

| 목표 | LLM 이 고른 것 |
|---|---|
| admin 계정이 로그인에 몇 번 실패했는지 알려 주세요. | `count_failed_logins` · `{"user": "admin"}` |
| 185.220.101.34 는 어느 나라의 주소인지 알려 주세요. | `lookup_ip` · `{"ip": "185.220.101.34"}` |
| admin 계정을 잠가 주세요. | `count_failed_logins` · `{"user": "admin"}` |
| 오늘 점심 메뉴를 추천해 주세요. | 「관련된 도구가 없습니다」 라는 문장, 또는 `{"tool": null, "args": {}}` |

1. 목표에 맞는 도구가 목록에 있으면 그것을 고릅니다.
2. **목록에 없는 일**(계정 잠금)을 시켰더니, 있는 도구 가운데 하나를 억지로 골랐습니다. 목표는 이뤄지지 않습니다.
3. 도구와 상관없는 목표에는 JSON 이 아닌 문장이 올 때도 있습니다. `parse_llm_json` 이 `None` 을 돌려주는 경우입니다.
4. 그래서 LLM 이 고른 것을 **그대로 실행하지 않고 확인**합니다. 6교시와 7교시가 그 확인입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-7 · IP 를 조회하는 도구 만들기</font></h3></td></tr></table>

IP 주소를 받아 **나라와 통신사**를 글자로 돌려주는 도구 `lookup_ip` 를 완성하시오. 2교시에 쓴 조회 API 를 씁니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Germany / Stiftung Erneuerbare Freiheit` |

**💡 힌트**

1. 조회 주소는 `"https://ipwho.is/"` 뒤에 `args["ip"]` 를 이어 붙여 만듭니다.
2. 응답을 딕셔너리로 바꾸는 것까지 한 줄에 씁니다. 2교시 문제 2-8 의 `truth` 줄과 같은 모양입니다.
3. 나라는 `country` 칸에, 통신사는 `connection` 안의 `isp` 칸에 있습니다.

In [ ]:
import requests


def lookup_ip(args):
    # 1. data 라는 변수를 만들어 "https://ipwho.is/" 뒤에 args 의 ip 를 붙인 주소로 GET 요청을 보내고, 응답을 딕셔너리로 바꾼 것을 담으세요 (timeout=10)


    # 2. data 의 country 와 connection 안의 isp 를 " / " 로 이어 붙여 return 하세요



print(lookup_ip({"ip": "185.220.101.34"}))

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-8 · 두 도구 가운데 고른 것을 실행하기</font></h3></td></tr></table>

LLM 이 고른 도구가 `count_failed_logins` 이면 그 함수를, `lookup_ip` 이면 그 함수를 실행해 결과를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `실행 결과: South Korea / SamsungSDS Inc.` |

**💡 힌트**

1. 갈래가 둘이므로 `if` 와 `elif` 를 씁니다.
2. 두 갈래 모두 `result` 라는 변수에 함수의 결과를 담습니다.
3. 2교시에 LLM 은 이 IP 의 통신사를 지어냈습니다. 이번에는 조회 도구가 실제 값을 가져옵니다.

In [ ]:
import json
import requests
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]

tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "211.45.12.9 는 어느 나라의 어느 통신사 주소인지 알려 주세요."
text = call_llm(tools_text + "목표: " + goal)
choice = parse_llm_json(text)
# ── 여기까지 ──

# 1. choice 의 tool 이 "count_failed_logins" 이면 result 에 count_failed_logins 의 결과를 담으세요 (두 줄)

# 2. choice 의 tool 이 "lookup_ip" 이면 result 에 lookup_ip 의 결과를 담으세요 (elif 로 두 줄)


print("실행 결과:", result)

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.4 실행 결과를 다시 LLM 에 넘겨 답을 만듭니다</mark>

도구가 돌려준 값은 숫자나 짧은 글자입니다. 그 값을 다시 LLM 에 넘겨 **사람이 읽을 문장**으로 만듭니다.

```python
prompt2 = "목표: admin 계정이 로그인에 몇 번 실패했는지 알려 주세요. "
prompt2 = prompt2 + "도구 count_failed_logins 를 실행한 결과는 4 입니다. "
prompt2 = prompt2 + "이 결과로 목표에 한 문장으로 답하세요."

print(call_llm(prompt2))          # admin 계정의 로그인 실패 횟수는 총 4번입니다.
```

1. 목표 → 도구 고르기 → 실행 → 결과로 답하기. 이 한 바퀴가 에이전트의 기본 흐름입니다.
2. 숫자 `4` 는 LLM 이 지어낸 것이 아니라 **우리 도구가 로그 파일에서 센 값**입니다.
3. 숫자를 글자에 이어 붙일 때는 9/22 에 배운 `str()` 로 글자로 바꿉니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-9 · 실행 결과로 최종 답 만들기</font></h3></td></tr></table>

도구를 실행해 얻은 `result` 를 LLM 에 다시 넘겨, 목표에 대한 **한 문장 답**을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `kim.cs 계정의 로그인 실패 횟수는 1번입니다.` 와 비슷한 한 문장 |

**💡 힌트**

1. 두 번째 질문 글자에는 목표, 실행 결과, 「한 문장으로 답하세요」 를 넣습니다.
2. `result` 는 숫자입니다. 글자에 이어 붙이려면 `str(result)` 로 바꿉니다.
3. 이번 호출은 도구를 고르는 것이 아니라 문장을 만드는 것입니다. `tools_text` 를 넣지 않습니다.

In [ ]:
import json
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
result = count_failed_logins({"user": "kim.cs"})      # LLM 이 고른 도구를 실행했다고 칩니다
# ── 여기까지 ──

# 1. prompt2 라는 변수를 만들어 "목표: " + goal 뒤에 " 도구를 실행한 결과는 " 과 str(result) 를 이어 붙이세요


# 2. prompt2 뒤에 " 입니다. 이 결과로 목표에 한 문장으로 답하세요." 를 이어 붙여 다시 prompt2 에 담으세요


# 3. final 이라는 변수를 만들어 call_llm 에 prompt2 를 넣어 받은 답을 담으세요


print(final)

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-10 · 목표 두 개를 차례로 처리하기</font></h3></td></tr></table>

`goals` 의 목표 둘을 차례로 처리하시오. 목표마다 LLM 이 도구를 고르고, 우리 코드가 실행하고, `고른 도구 → 결과` 꼴로 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `count_failed_logins → 4` 그리고 `lookup_ip → Germany / Stiftung Erneuerbare Freiheit` |

**💡 힌트**

1. 반복 안에서 호출, 딕셔너리로 바꾸기, `if`·`elif` 로 실행하기를 차례로 합니다. 문제 5-8 과 같은 방법입니다.
2. 질문 글자는 `tools_text + "목표: " + goal` 입니다.
3. 출력은 `print(choice["tool"], "→", result)` 로 합니다.

In [ ]:
import json
import requests
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]

tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "185.220.101.34 는 어느 나라의 주소인지 알려 주세요.",
]
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 text 에 call_llm 의 답을, choice 에 parse_llm_json 의 결과를 담으세요
# 3. choice 의 tool 에 따라 if · elif 로 알맞은 도구를 실행해 result 에 담으세요
# 4. choice 의 tool, "→", result 를 출력하세요

### 📋 5교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 도구를 만든다 | `def count_failed_logins(args):` … `return count` |
| LLM 에게 고르게 한다 | `call_llm(tools_text + "목표: " + goal)` |
| 고른 것을 읽는다 | `choice = parse_llm_json(text)` · `choice["tool"]` · `choice["args"]` |
| 우리 코드로 실행한다 | `if choice["tool"] == "…":` `result = 도구(choice["args"])` |
| 결과로 답을 만든다 | `call_llm("목표: … 실행한 결과는 … 한 문장으로 답하세요.")` |

LLM 은 고르기만 합니다. 실행은 우리 코드가 합니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · 라우터 — 이름으로 함수를 찾아 실행한다</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **라우터 (router)** | 프로그램에서 「라우팅」 은 무엇을 무엇으로 이어 주는 일인가 |
| **레지스트리 (registry)** | 이름과 실제 대상을 짝지어 두면 무엇이 편해지는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 라우터 →
- 레지스트리 →

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6 · 이름으로 함수를 찾아 실행한다</mark>

### 왜 필요한가

1. 5교시에는 도구마다 `if` · `elif` 를 한 갈래씩 썼습니다. 도구가 열 개면 갈래도 열 개입니다.
2. LLM 이 **목록에 없는 이름**을 답하면 어느 갈래에도 걸리지 않고 조용히 지나갑니다. 왜 아무 일도 안 일어났는지 아무도 모릅니다.
3. 그래서 **이름과 함수를 짝지은 딕셔너리**를 만들고, 이름으로 함수를 찾아 실행하는 코드를 한 곳에 둡니다. 없는 이름이면 그 자리에서 알립니다.

### 이 시간에 나오는 말

| 말 | 뜻 | 6교시에서 보는 모양 |
|---|---|---|
| 레지스트리 | 이름과 함수를 짝지어 둔 딕셔너리 | `tool_registry = {"lookup_ip": lookup_ip}` |
| 라우터 | 이름을 보고 실제 함수를 찾아 실행하는 코드 | `route_tool_call(choice)` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 함수도 딕셔너리의 값으로 담을 수 있습니다</mark>

함수 이름을 **괄호 없이** 쓰면 실행하지 않고 함수 자체를 가리킵니다. 그래서 딕셔너리의 값으로 담을 수 있습니다.

```python
def say_ok(args):
    return "ok " + args["name"]

registry = {"ok": say_ok}          # 괄호가 없다 — 실행하지 않고 담기만 한다

func = registry.get("ok")           # 이름으로 함수를 꺼낸다
print(func({"name": "deploy"}))     # 꺼낸 함수를 실행한다 → ok deploy

print(registry.get("없는이름"))      # None
```

1. `say_ok` 는 함수 자체이고, `say_ok(…)` 는 함수를 실행한 결과입니다. 10/2 오후 스케줄러에 함수를 넘길 때도 괄호 없이 썼습니다.
2. `.get()` 은 9/22 에 배웠습니다. 키가 없으면 오류를 내지 않고 `None` 을 돌려줍니다.
3. 도구가 늘어도 **딕셔너리에 한 줄만 더하면** 됩니다. 실행하는 코드는 바뀌지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
def double(args):
    return args["n"] * 2

registry = {"double": double}
func = registry.get("double")

print(func({"n": 21}))
```

막히면 바로 위 `6.1 함수도 딕셔너리의 값으로 담을 수 있습니다` 설명을 다시 봅니다.

In [ ]:
def double(args):
    return args["n"] * 2

registry = {"double": double}
func = registry.get("double")

print(func({"n": 21}))

✅ `42` — 이름 `double` 로 함수를 꺼내고, 꺼낸 함수를 실행했습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
def double(args):
    return args["n"] * 2

registry = {"double": double}

print(registry.get("triple"))
```

막히면 바로 위 `6.1 함수도 딕셔너리의 값으로 담을 수 있습니다` 설명을 다시 봅니다.

In [ ]:
def double(args):
    return args["n"] * 2

registry = {"double": double}

print(registry.get("triple"))

✅ `None` — `triple` 이라는 이름은 등록되어 있지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · 도구 두 개를 레지스트리에 등록하기</font></h3></td></tr></table>

빈 딕셔너리 `tool_registry` 를 만들고, 도구 두 개를 **이름과 함수의 짝**으로 등록하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `2` |

**💡 힌트**

1. 빈 딕셔너리는 `{}` 입니다.
2. 키는 도구 이름 글자이고, 값은 **괄호 없는** 함수 이름입니다.
3. 키 이름은 LLM 에게 알려 준 도구 이름과 한 글자도 다르면 안 됩니다.

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
# ── 여기까지 ──

# 1. tool_registry 라는 변수를 만들어 빈 딕셔너리를 담으세요


# 2. tool_registry 의 "count_failed_logins" 칸에 함수 count_failed_logins 를 담으세요


# 3. tool_registry 의 "lookup_ip" 칸에 함수 lookup_ip 를 담으세요


print(len(tool_registry))

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-4 · 이름으로 함수를 꺼내 실행하기</font></h3></td></tr></table>

`choice` 는 LLM 이 고른 결과입니다. 레지스트리에서 **이름으로 함수를 꺼내** 실행하고 결과를 출력하시오. `if` · `elif` 를 쓰지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `4` |

**💡 힌트**

1. 도구 이름은 `choice["tool"]` 에 있습니다.
2. 레지스트리에서 꺼낼 때는 `.get()` 을 씁니다.
3. 꺼낸 함수에 `choice["args"]` 를 넣어 실행합니다.

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
choice = {"tool": "count_failed_logins", "args": {"user": "admin"}}
# ── 여기까지 ──

# 1. name 이라는 변수를 만들어 choice 의 tool 을 담으세요


# 2. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요


# 3. result 라는 변수를 만들어 func 에 choice 의 args 를 넣어 실행한 값을 담으세요


print(result)

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.2 등록되지 않은 이름은 조용히 넘기지 않습니다</mark>

5교시에 봤듯이 LLM 은 목록에 없는 일을 시키면 엉뚱한 도구를 고르거나, 없는 이름을 답할 수 있습니다.

```python
func = tool_registry.get("delete_all_logs")          # None

if func:
    result = func(args)
else:
    print("[오류] 등록되지 않은 도구: delete_all_logs")
```

1. 레지스트리에 없는 이름이면 `.get()` 이 `None` 을 돌려줍니다. `if func:` 로 확인합니다.
2. 없는 이름을 **무시하지 않고 알립니다.** LLM 이 도구를 지어냈거나 레지스트리를 잘못 만든 것이라, 어느 쪽이든 사람이 알아야 합니다.
3. 조용히 넘어가면 「왜 조치가 안 됐는지」 가 어디에도 남지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · 등록되지 않은 도구를 알리기</font></h3></td></tr></table>

`choice` 는 레지스트리에 **없는 이름**을 담고 있습니다. 함수를 찾았으면 실행해 출력하고, 못 찾았으면 `[오류] 등록되지 않은 도구:` 와 이름을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[오류] 등록되지 않은 도구: delete_all_logs` |

**💡 힌트**

1. 이름으로 꺼내는 줄은 문제 6-4 와 같습니다.
2. 꺼낸 것이 있는지는 `if func:` 로 확인합니다.
3. `else` 아래에서 오류 문장과 `name` 을 출력합니다.

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
choice = {"tool": "delete_all_logs", "args": {}}
name = choice["tool"]
# ── 여기까지 ──

# 1. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요


# 2. func 가 있으면 func 에 choice 의 args 를 넣어 실행한 값을 출력하고, 없으면 "[오류] 등록되지 않은 도구:" 와 name 을 출력하세요 (네 줄)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-6 · route_tool_call 함수 만들기</font></h3></td></tr></table>

LLM 이 고른 `choice` 를 받아, 레지스트리에서 함수를 찾아 실행하고 결과를 돌려주는 함수 `route_tool_call` 을 완성하시오. 없는 이름이면 오류를 알리고 `None` 을 돌려줍니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1` 그리고 `[오류] 등록되지 않은 도구: block_everything` 그리고 `None` |

**💡 힌트**

1. 함수 안의 줄은 네 칸 들여 씁니다.
2. 찾았으면 실행한 값을 `return` 으로 돌려줍니다.
3. 못 찾았을 때의 두 줄은 미리 채워져 있습니다.

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
# ── 여기까지 ──


def route_tool_call(choice):
    # 1. name 이라는 변수를 만들어 choice 의 tool 을 담으세요


    # 2. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요


    # 3. func 가 있으면 func 에 choice 의 args 를 넣어 실행한 값을 return 하세요 (두 줄)


    print("[오류] 등록되지 않은 도구:", name)
    return None


print(route_tool_call({"tool": "count_failed_logins", "args": {"user": "park.js"}}))
print(route_tool_call({"tool": "block_everything", "args": {}}))

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-7 · tool_router.py 만들기</font></h3></td></tr></table>

오후의 산출물 **`tool_router.py`** 를 만드시오. 도구 둘, 레지스트리, 라우터를 파일 하나에 담습니다. 문제 6-3 과 6-6 에서 쓴 줄을 다시 씁니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Writing tool_router.py` (다시 실행하면 `Overwriting tool_router.py`) |

**💡 힌트**

1. 레지스트리는 도구 함수를 정의한 **뒤에** 만들어야 합니다. 함수가 먼저 있어야 담을 수 있습니다.
2. 등록하는 두 줄은 문제 6-3 과, 라우터 안의 줄은 문제 6-6 과 같습니다.
3. 빈 줄을 다 채운 뒤에 실행합니다.

In [ ]:
%%writefile tool_router.py
import json
import requests


def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count


def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]


tool_registry = {}
# 1. tool_registry 에 도구 두 개를 이름과 함수의 짝으로 등록하세요 (두 줄)



def route_tool_call(choice):
    name = choice["tool"]
    # 2. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요


    # 3. func 가 있으면 func 에 choice 의 args 를 넣어 실행한 값을 return 하세요 (두 줄)


    print("[오류] 등록되지 않은 도구:", name)
    return None

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-8 · LLM 이 고른 도구를 라우터로 실행하기</font></h3></td></tr></table>

`llm_client.py` 와 `tool_router.py` 를 이어, 목표 하나를 처리하시오. LLM 이 도구를 고르고 라우터가 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `count_failed_logins → 1` |

**💡 힌트**

1. 호출과 딕셔너리로 바꾸기는 5교시와 같습니다.
2. `choice` 가 `None` 일 수 있으므로 `if choice:` 로 확인한 뒤에 라우터에 넘깁니다.
3. 5교시의 `if` · `elif` 갈래가 `route_tool_call(choice)` 한 줄로 바뀌었습니다.

In [ ]:
from llm_client import call_llm, parse_llm_json
from tool_router import route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "park.js 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. text 라는 변수를 만들어 call_llm 에 tools_text + "목표: " + goal 을 넣어 받은 답을 담으세요


# 2. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요


# 3. choice 가 있으면 result 에 route_tool_call(choice) 의 결과를 담고, choice 의 tool, "→", result 를 출력하세요 (세 줄)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-9 · LLM 이 도구를 고르지 못했을 때</font></h3></td></tr></table>

도구와 상관없는 목표를 보냅니다. LLM 의 답을 읽지 못했으면 `도구를 고르지 못했습니다` 를, 읽었으면 라우터에 넘겨 결과를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `도구를 고르지 못했습니다`, 또는 `[오류] 등록되지 않은 도구: None` 과 `None`. LLM 이 문장으로 답하면 앞의 것이, `{"tool": null}` 로 답하면 뒤의 것이 나옵니다 |

**💡 힌트**

1. 딕셔너리로 바꾸기는 문제 6-8 과 같습니다.
2. `choice` 가 있으면 라우터에 넘기고, 없으면 `else` 에서 문장을 출력합니다.
3. 어느 쪽이 나와도 **프로그램은 멈추지 않고, 아무 도구도 실행되지 않았습니다.**

In [ ]:
from llm_client import call_llm, parse_llm_json
from tool_router import route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "오늘 점심 메뉴를 추천해 주세요."
text = call_llm(tools_text + "목표: " + goal)
# ── 여기까지 ──

# 1. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요


# 2. choice 가 있으면 route_tool_call(choice) 의 결과를 출력하고, 없으면 "도구를 고르지 못했습니다" 를 출력하세요 (네 줄)

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-10 · 라우터를 고치지 않고 도구를 하나 더하기</font></h3></td></tr></table>

IP 하나에서 온 기록이 몇 건인지 세는 도구 `count_by_ip` 를 만들어 레지스트리에 등록하고, 라우터로 실행하시오. **`route_tool_call` 은 고치지 않습니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `5` |

**💡 힌트**

1. 도구의 모양은 `count_failed_logins` 와 같습니다. 견주는 칸이 `user` 가 아니라 `ip` 이고, 등급은 따지지 않습니다.
2. 등록은 `tool_registry["count_by_ip"] = count_by_ip` 한 줄입니다.
3. 등록한 뒤에 `route_tool_call` 에 `choice` 를 넘깁니다.

In [ ]:
import json
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
choice = {"tool": "count_by_ip", "args": {"ip": "211.45.12.9"}}
# ── 여기까지 ──

# 1. count_by_ip 라는 함수를 만드세요. args 를 받아, normalized_logs.json 에서 ip 가 args 의 ip 와 같은 기록의 수를 return 합니다
# 2. tool_registry 의 "count_by_ip" 칸에 함수 count_by_ip 를 담으세요
# 3. route_tool_call 에 choice 를 넣어 받은 결과를 출력하세요

### 📋 6교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 도구를 등록한다 | `tool_registry["lookup_ip"] = lookup_ip` (괄호 없이) |
| 이름으로 함수를 꺼낸다 | `func = tool_registry.get(name)` |
| 꺼낸 함수를 실행한다 | `func(choice["args"])` |
| 없는 이름을 알린다 | `print("[오류] 등록되지 않은 도구:", name)` |
| 한 줄로 실행한다 | `route_tool_call(choice)` |

오후 산출물 `agent_core/tool_router.py` 가 생겼습니다. 도구가 늘어도 라우터는 고치지 않습니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · 승인 게이트 — 위험한 도구 앞에 사람을 둔다</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **사람 개입 (human-in-the-loop)** | 자동화 흐름 가운데 어디에 사람을 두는가 |
| **파이썬 `input()`** | 프로그램이 실행 도중에 사람의 입력을 받는 방법 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 사람 개입 →
- input() →

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7 · 위험한 도구 앞에 사람을 둔다</mark>

### 왜 필요한가

1. 로그를 세거나 IP 를 조회하는 도구는 틀려도 다시 하면 됩니다.
2. **계정 잠금**은 다릅니다. 잘못 잠그면 그 사람은 일을 못 하고, 되돌리는 데 사람 손이 듭니다.
3. 5교시에 봤듯이 LLM 은 엉뚱한 도구를 고를 수 있습니다. 그래서 **되돌리기 어려운 도구 앞에는 사람의 확인**을 둡니다. 이것을 승인 게이트라고 합니다.

### 이 시간에 나오는 말

| 말 | 뜻 | 7교시에서 보는 모양 |
|---|---|---|
| 승인 게이트 | 위험한 실행 앞에 사람 확인을 두는 것 | `if choice["tool"] in need_approval:` |
| 사람 개입 | 자동 흐름 중간에 사람이 끼는 설계 | `answer = input("실행할까요? (y/n) ")` |
| `input()` | 실행 도중에 사람이 친 글자를 받는 함수 | `answer = input("…")` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.1 승인이 필요한 도구를 목록으로 정해 둡니다</mark>

도구를 **되돌릴 수 있는가**로 나눕니다. 승인이 필요한 도구의 이름을 리스트에 적어 둡니다.

```python
need_approval = ["lock_account"]

print("lock_account" in need_approval)          # True
print("lookup_ip" in need_approval)             # False
```

| 바로 실행해도 되는 도구 | 승인을 받아야 하는 도구 |
|---|---|
| `count_failed_logins` · `lookup_ip` | `lock_account` |
| 틀려도 다시 하면 된다 | 잘못하면 사람이 일을 못 한다 |

1. `in` 은 9/23 에 배웠습니다. 리스트 안에 그 값이 있으면 `True` 입니다.
2. 승인 여부를 LLM 이 정하게 하지 않습니다. **규칙은 코드에 적혀 있어야** 언제나 같게 동작합니다.
3. 오늘의 `lock_account` 는 실습이라 화면에 출력만 합니다. 실제 계정을 잠그지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
need_approval = ["lock_account", "block_ip"]

print("block_ip" in need_approval)
print("count_failed_logins" in need_approval)
```

막히면 바로 위 `7.1 승인이 필요한 도구를 목록으로 정해 둡니다` 설명을 다시 봅니다.

In [ ]:
need_approval = ["lock_account", "block_ip"]

print("block_ip" in need_approval)
print("count_failed_logins" in need_approval)

✅ `True` 그리고 `False`

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-2 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
answer = "n"

if answer == "y":
    print("실행합니다")
else:
    print("보류합니다")
```

막히면 바로 위 `7.1 승인이 필요한 도구를 목록으로 정해 둡니다` 설명을 다시 봅니다.

In [ ]:
answer = "n"

if answer == "y":
    print("실행합니다")
else:
    print("보류합니다")

✅ `보류합니다` — `y` 가 아니면 실행하지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-3 · 계정 잠금 도구를 만들고 등록하기</font></h3></td></tr></table>

계정 잠금 도구 `lock_account` 를 완성하고 레지스트리에 등록하시오. 실습이므로 화면에 출력만 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[조치] 계정 잠금: guest01` 그리고 `locked` |

**💡 힌트**

1. 잠글 계정은 `args["user"]` 에 있습니다.
2. 도구는 실행 결과를 돌려줘야 합니다. 여기서는 글자 `"locked"` 를 돌려줍니다.
3. 등록은 6교시와 같은 방법입니다. 괄호 없이 함수 이름을 담습니다.

In [ ]:
from tool_router import tool_registry, route_tool_call


def lock_account(args):
    # 1. "[조치] 계정 잠금:" 과 args 의 user 를 출력하세요


    # 2. "locked" 를 return 하세요



# 3. tool_registry 의 "lock_account" 칸에 함수 lock_account 를 담으세요


print(route_tool_call({"tool": "lock_account", "args": {"user": "guest01"}}))

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-4 · 승인이 필요한 도구인지 가리기</font></h3></td></tr></table>

`choices` 의 선택 둘을 하나씩 보고, 승인이 필요하면 `승인 필요`, 아니면 `바로 실행` 을 도구 이름과 함께 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `lookup_ip 바로 실행` 그리고 `lock_account 승인 필요` |

**💡 힌트**

1. 고른 도구의 이름은 `choice["tool"]` 입니다.
2. 그 이름이 `need_approval` 안에 있는지는 `in` 으로 확인합니다.
3. 출력은 이름과 문구를 쉼표로 나란히 넣습니다.

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
need_approval = ["lock_account"]
choices = [
    {"tool": "lookup_ip", "args": {"ip": "198.51.100.23"}},
    {"tool": "lock_account", "args": {"user": "root"}},
]
# ── 여기까지 ──

for choice in choices:
    # 1. choice 의 tool 이 need_approval 안에 있으면 choice 의 tool 과 "승인 필요" 를 출력하세요 (두 줄)

    # 2. 없으면 choice 의 tool 과 "바로 실행" 을 출력하세요 (두 줄)

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.2 input() 으로 사람에게 묻습니다</mark>

`input()` 은 프로그램을 잠깐 멈추고 **사람이 글자를 칠 때까지 기다립니다.** 친 글자를 돌려줍니다.

```python
answer = input("lock_account 을 실행할까요? (y/n) ")

if answer == "y":
    print("실행합니다")
else:
    print("보류합니다")
```

1. 괄호 안의 글자는 사람에게 보여 줄 질문입니다.
2. VS Code 에서는 셀을 실행하면 **화면 위쪽에 입력 칸**이 뜹니다. 글자를 치고 Enter 를 누릅니다.
3. 받은 값은 글자입니다. `"y"` 와 같은지 `==` 로 견줍니다.
4. `y` 가 아닌 모든 답은 실행하지 않는 쪽으로 처리합니다. 잘못 누른 것이 실행으로 이어지면 안 됩니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-5 · 사람에게 묻고 승인하면 실행하기</font></h3></td></tr></table>

`lock_account` 를 실행하기 전에 사람에게 묻습니다. `y` 를 받으면 라우터로 실행하고, 아니면 `result` 에 `"held"` 를 담으시오. **한 번은 `y`, 한 번은 `n` 을 넣어 두 번 실행해 봅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `y` 를 넣으면 `[조치] 계정 잠금: svc_backup` 과 `locked`, `n` 을 넣으면 `held` |

**💡 힌트**

1. 질문 글자는 `"lock_account 을 실행할까요? (y/n) "` 입니다.
2. `y` 이면 `route_tool_call(choice)` 의 결과를 `result` 에 담습니다.
3. `y` 가 아니면 `result` 에 `"held"` 를 담습니다. 보류했다는 뜻입니다.

In [ ]:
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]
choice = {"tool": "lock_account", "args": {"user": "svc_backup"}}
# ── 여기까지 ──

# 1. answer 라는 변수를 만들어 input 으로 "lock_account 을 실행할까요? (y/n) " 를 물어 받은 글자를 담으세요


# 2. answer 가 "y" 이면 result 에 route_tool_call(choice) 의 결과를, 아니면 result 에 "held" 를 담으세요 (네 줄)


print(result)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-6 · 승인 게이트를 함수로 만들기</font></h3></td></tr></table>

`choice` 를 받아, 승인이 필요한 도구이면 사람에게 묻고 실행하는 함수 `run_with_gate` 를 완성하시오. 승인하지 않으면 `"held"` 를 돌려줍니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1` 이 먼저 나오고, 그다음 물음에 `n` 을 넣으면 `held` |

**💡 힌트**

1. 승인이 필요한 도구인지는 문제 7-4 와 같은 방법으로 확인합니다.
2. `y` 가 아니면 그 자리에서 `return "held"` 로 함수를 끝냅니다. `!=` 는 「같지 않다」 입니다.
3. 승인이 필요 없거나 승인을 받았으면 마지막 줄에서 라우터로 실행합니다.

In [ ]:
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]
# ── 여기까지 ──


def run_with_gate(choice):
    # 1. choice 의 tool 이 need_approval 안에 있으면 (if 한 줄)

        # 2. answer 라는 변수를 만들어 input 으로 choice 의 tool + " 을 실행할까요? (y/n) " 를 물어 받은 글자를 담으세요

        # 3. answer 가 "y" 가 아니면 "held" 를 return 하세요 (두 줄)


    # 4. route_tool_call(choice) 의 결과를 return 하세요



print(run_with_gate({"tool": "count_failed_logins", "args": {"user": "lee.yh"}}))
print(run_with_gate({"tool": "lock_account", "args": {"user": "lee.yh"}}))

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.3 무엇을 실행했는지 기록으로 남깁니다</mark>

에이전트가 한 일은 **실행했든 보류했든** 기록으로 남깁니다. 9/28 오후에 배운 `json.dump` 로 `agent_result.json` 에 저장합니다.

```python
records = []
records.append({"goal": "admin 계정이 몇 번 실패했나", "tool": "count_failed_logins", "result": 4})

with open("agent_result.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
```

1. 기록 한 건은 딕셔너리입니다. **목표 · 고른 도구 · 결과**를 담습니다.
2. 보류한 것도 `"result": "held"` 로 남깁니다. 남기지 않으면 「왜 조치가 안 됐는지」 를 설명할 수 없습니다.
3. `ensure_ascii=False` 는 한글을 그대로 저장하고, `indent=2` 는 두 칸 들여쓰기입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-7 · 실행한 일을 기록에 더하기</font></h3></td></tr></table>

`choice` 를 라우터로 실행하고, **목표 · 도구 · 결과**를 딕셔너리로 만들어 `records` 에 더하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[{'goal': 'jung.hw 계정이 몇 번 실패했나', 'tool': 'count_failed_logins', 'result': 1}]` |

**💡 힌트**

1. 결과는 `route_tool_call(choice)` 로 얻습니다.
2. 딕셔너리의 키는 `goal`, `tool`, `result` 셋입니다.
3. 리스트에 더하는 것은 9/22 에 배운 `.append()` 입니다.

In [ ]:
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "jung.hw 계정이 몇 번 실패했나"
choice = {"tool": "count_failed_logins", "args": {"user": "jung.hw"}}
records = []
# ── 여기까지 ──

# 1. result 라는 변수를 만들어 route_tool_call 에 choice 를 넣어 받은 값을 담으세요


# 2. record 라는 변수를 만들어 goal, choice 의 tool, result 를 담은 딕셔너리를 만드세요. 키는 goal, tool, result 입니다


# 3. records 에 record 를 더하세요


print(records)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-8 · 기록을 agent_result.json 으로 저장하기</font></h3></td></tr></table>

`records` 를 오후의 산출물 **`agent_result.json`** 으로 저장하고, 다시 읽어 건수를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `2건 저장했습니다` |

**💡 힌트**

1. 쓰기로 열 때는 `"w"` 를 넣습니다. 한글이 들어가므로 `encoding="utf-8"` 도 넣습니다.
2. 저장은 `json.dump` 입니다. 한글 그대로, 두 칸 들여쓰기로 저장합니다.
3. 다시 읽는 것은 9/28 오후에 배운 `json.load` 입니다.

In [ ]:
import json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
records = [
    {"goal": "admin 계정이 몇 번 실패했나", "tool": "count_failed_logins", "result": 4},
    {"goal": "admin 계정을 잠가 달라", "tool": "lock_account", "result": "held"},
]
# ── 여기까지 ──

# 1. "agent_result.json" 을 쓰기로 여세요. 변수 이름은 f 입니다 (with 한 줄)

    # 2. records 를 f 에 JSON 으로 저장하세요. 한글 그대로, 두 칸 들여쓰기입니다


# 3. 저장한 파일을 다시 읽기로 열어, saved 라는 변수에 읽은 내용을 담으세요 (두 줄)


print(f"{len(saved)}건 저장했습니다")

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-9 · 목표 하나를 처음부터 끝까지 처리하기</font></h3></td></tr></table>

목표 하나를 **고르기 → 게이트 → 실행 → 기록**까지 한 번에 처리하시오. LLM 이 `lock_account` 를 고르면 물음이 뜹니다. `y` 를 넣습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[조치] 계정 잠금: admin` 그리고 `[{'goal': …, 'tool': 'lock_account', 'result': 'locked'}]` |

**💡 힌트**

1. 호출과 딕셔너리로 바꾸기는 6교시와 같습니다.
2. 라우터 대신 **게이트 함수** `run_with_gate` 에 `choice` 를 넘깁니다.
3. 기록은 문제 7-7 과 같은 딕셔너리입니다.

In [ ]:
from llm_client import call_llm, parse_llm_json
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]

def run_with_gate(choice):
    if choice["tool"] in need_approval:
        answer = input(choice["tool"] + " 을 실행할까요? (y/n) ")
        if answer != "y":
            return "held"
    return route_tool_call(choice)

tools_text = "쓸 수 있는 도구는 셋입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "3) lock_account: 계정을 잠근다. 인자는 user. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요."
records = []
# ── 여기까지 ──

# 1. text 라는 변수에 call_llm(tools_text + "목표: " + goal) 의 답을, choice 라는 변수에 parse_llm_json(text) 의 결과를 담으세요 (두 줄)


# 2. choice 가 있으면 result 에 run_with_gate(choice) 의 결과를 담고, goal · tool · result 를 담은 딕셔너리를 records 에 더하세요 (세 줄)


print(records)

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-10 · 목표 세 개를 처리하는 작은 에이전트</font></h3></td></tr></table>

`goals` 의 목표 셋을 차례로 처리해 `agent_result.json` 에 저장하시오. 잠금 물음이 뜨면 `n` 을 넣어 보류합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `3건 저장했습니다` — 파일을 열면 `count_failed_logins` · `lookup_ip` · `lock_account`(`held`) 세 건이 있습니다 |

**💡 힌트**

1. 반복 안의 줄은 문제 7-9 와 같습니다.
2. 반복이 끝난 뒤에 한 번만 저장합니다. 저장은 문제 7-8 과 같습니다.
3. 보류한 것도 기록에 남는지 `agent_result.json` 을 열어 확인합니다.

In [ ]:
import json
from llm_client import call_llm, parse_llm_json
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]

def run_with_gate(choice):
    if choice["tool"] in need_approval:
        answer = input(choice["tool"] + " 을 실행할까요? (y/n) ")
        if answer != "y":
            return "held"
    return route_tool_call(choice)

tools_text = "쓸 수 있는 도구는 셋입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "3) lock_account: 계정을 잠근다. 인자는 user. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "211.45.12.9 는 어느 나라의 주소인지 알려 주세요.",
    "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요.",
]
records = []
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 text 와 choice 를 만드세요
# 3. choice 가 있으면 run_with_gate 로 실행하고, goal · tool · result 를 records 에 더하세요
# 4. 반복이 끝나면 records 를 "agent_result.json" 으로 저장하세요


print(f"{len(records)}건 저장했습니다")

### 📋 7교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 승인이 필요한 도구를 정한다 | `need_approval = ["lock_account"]` |
| 승인이 필요한지 확인한다 | `if choice["tool"] in need_approval:` |
| 사람에게 묻는다 | `answer = input("… (y/n) ")` |
| 승인하지 않으면 보류한다 | `if answer != "y":` `return "held"` |
| 한 일을 기록한다 | `records.append({"goal": …, "tool": …, "result": …})` |
| 파일로 남긴다 | `json.dump(records, f, ensure_ascii=False, indent=2)` |

판단은 LLM 이 하고, 실행은 우리 코드가 하고, 되돌리기 어려운 결정은 사람이 합니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 오늘 만든 것 확인 · 남은 도전</mark>

새로 배우는 것은 없습니다. 오늘 만든 파일이 `agent_core` 폴더에 있는지 확인하고, 건너뛴 도전 문제를 풉니다.

아래 셀을 실행하면 오늘의 산출물 셋이 있는지 알려 줍니다.

In [ ]:
import os

for name in ["llm_client.py", "tool_router.py", "agent_result.json"]:
    if os.path.exists(name):
        print("있음 :", name)
    else:
        print("없음 :", name, "— 그 파일을 만드는 문제로 돌아갑니다")

### 오늘 만든 것

| 파일 | 만든 문제 | 하는 일 |
|---|---|---|
| `llm_client.py` | 오전 4-7 | LLM 에 질문을 보내고, 답을 딕셔너리로 읽는다 |
| `tool_router.py` | 오후 6-7 | 도구를 등록하고, 이름으로 찾아 실행한다 |
| `agent_result.json` | 오후 7-8 · 7-10 | 에이전트가 한 일의 기록 |

### 남은 도전 문제

| 도전 | 무엇을 해 보나 |
|---|---|
| ⭐5-10 | 목표 두 개를 차례로 처리한다 |
| ⭐6-10 | 라우터를 고치지 않고 도구를 하나 더한다 |
| ⭐7-10 | 목표 세 개를 처리해 `agent_result.json` 에 남긴다 |

기본 문제를 못 끝냈으면 도전이 아니라 기본 문제부터 합니다.

### 내일 (10/7)

오늘은 로그 한 줄, 목표 하나씩 다뤘습니다. 내일은 **이벤트 수십 건을 나눠서 요약**하고, 그 결과로 일일 보고서를 만듭니다. 오늘 만든 `llm_client.py` 를 그대로 씁니다.

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 5-3

In [ ]:
import json


def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        # 1. row 의 user 가 args 의 user 와 같고 level 이 "WARN" 이면 (if 한 줄)
        if row["user"] == args["user"] and row["level"] == "WARN":
            # 2. count 에 1 을 더하세요
            count = count + 1

    # 3. count 를 return 하세요
    return count


print(count_failed_logins({"user": "admin"}))

### 정답 5-4

In [ ]:
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. prompt 라는 변수를 만들어 tools_text, "목표: ", goal 을 이어 붙이세요
prompt = tools_text + "목표: " + goal

# 2. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담으세요
text = call_llm(prompt)

# 3. text 를 출력하세요
print(text)

### 정답 5-5

In [ ]:
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. prompt 라는 변수를 만들어 tools_text, "목표: ", goal 을 이어 붙이세요
prompt = tools_text + "목표: " + goal

# 2. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담으세요
text = call_llm(prompt)

# 3. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요
choice = parse_llm_json(text)

# 4. choice 의 tool 칸과 args 칸을 한 줄씩 출력하세요 (두 줄)
print(choice["tool"])
print(choice["args"])

### 정답 5-6

In [ ]:
import json
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
text = call_llm(tools_text + "목표: " + goal)
choice = parse_llm_json(text)
# ── 여기까지 ──

# 1. choice 의 tool 이 "count_failed_logins" 이면 (if 한 줄)
if choice["tool"] == "count_failed_logins":
    # 2. result 라는 변수를 만들어 count_failed_logins 에 choice 의 args 를 넣어 받은 값을 담으세요
    result = count_failed_logins(choice["args"])
    # 3. "실행 결과:" 와 result 를 출력하세요
    print("실행 결과:", result)

### 정답 5-7

In [ ]:
import requests


def lookup_ip(args):
    # 1. data 라는 변수를 만들어 "https://ipwho.is/" 뒤에 args 의 ip 를 붙인 주소로 GET 요청을 보내고, 응답을 딕셔너리로 바꾼 것을 담으세요 (timeout=10)
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()

    # 2. data 의 country 와 connection 안의 isp 를 " / " 로 이어 붙여 return 하세요
    return data["country"] + " / " + data["connection"]["isp"]


print(lookup_ip({"ip": "185.220.101.34"}))

### 정답 5-8

In [ ]:
import json
import requests
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]

tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "211.45.12.9 는 어느 나라의 어느 통신사 주소인지 알려 주세요."
text = call_llm(tools_text + "목표: " + goal)
choice = parse_llm_json(text)
# ── 여기까지 ──

# 1. choice 의 tool 이 "count_failed_logins" 이면 result 에 count_failed_logins 의 결과를 담으세요 (두 줄)
if choice["tool"] == "count_failed_logins":
    result = count_failed_logins(choice["args"])
# 2. choice 의 tool 이 "lookup_ip" 이면 result 에 lookup_ip 의 결과를 담으세요 (elif 로 두 줄)
elif choice["tool"] == "lookup_ip":
    result = lookup_ip(choice["args"])

print("실행 결과:", result)

### 정답 5-9

In [ ]:
import json
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
result = count_failed_logins({"user": "kim.cs"})      # LLM 이 고른 도구를 실행했다고 칩니다
# ── 여기까지 ──

# 1. prompt2 라는 변수를 만들어 "목표: " + goal 뒤에 " 도구를 실행한 결과는 " 과 str(result) 를 이어 붙이세요
prompt2 = "목표: " + goal + " 도구를 실행한 결과는 " + str(result)

# 2. prompt2 뒤에 " 입니다. 이 결과로 목표에 한 문장으로 답하세요." 를 이어 붙여 다시 prompt2 에 담으세요
prompt2 = prompt2 + " 입니다. 이 결과로 목표에 한 문장으로 답하세요."

# 3. final 이라는 변수를 만들어 call_llm 에 prompt2 를 넣어 받은 답을 담으세요
final = call_llm(prompt2)

print(final)

### 정답 5-10

In [ ]:
import json
import requests
from llm_client import call_llm, parse_llm_json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]

tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "185.220.101.34 는 어느 나라의 주소인지 알려 주세요.",
]
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 text 에 call_llm 의 답을, choice 에 parse_llm_json 의 결과를 담으세요
# 3. choice 의 tool 에 따라 if · elif 로 알맞은 도구를 실행해 result 에 담으세요
# 4. choice 의 tool, "→", result 를 출력하세요
for goal in goals:
    text = call_llm(tools_text + "목표: " + goal)
    choice = parse_llm_json(text)
    if choice["tool"] == "count_failed_logins":
        result = count_failed_logins(choice["args"])
    elif choice["tool"] == "lookup_ip":
        result = lookup_ip(choice["args"])
    print(choice["tool"], "→", result)

### 정답 6-3

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
# ── 여기까지 ──

# 1. tool_registry 라는 변수를 만들어 빈 딕셔너리를 담으세요
tool_registry = {}

# 2. tool_registry 의 "count_failed_logins" 칸에 함수 count_failed_logins 를 담으세요
tool_registry["count_failed_logins"] = count_failed_logins

# 3. tool_registry 의 "lookup_ip" 칸에 함수 lookup_ip 를 담으세요
tool_registry["lookup_ip"] = lookup_ip

print(len(tool_registry))

### 정답 6-4

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
choice = {"tool": "count_failed_logins", "args": {"user": "admin"}}
# ── 여기까지 ──

# 1. name 이라는 변수를 만들어 choice 의 tool 을 담으세요
name = choice["tool"]

# 2. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요
func = tool_registry.get(name)

# 3. result 라는 변수를 만들어 func 에 choice 의 args 를 넣어 실행한 값을 담으세요
result = func(choice["args"])

print(result)

### 정답 6-5

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
choice = {"tool": "delete_all_logs", "args": {}}
name = choice["tool"]
# ── 여기까지 ──

# 1. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요
func = tool_registry.get(name)

# 2. func 가 있으면 func 에 choice 의 args 를 넣어 실행한 값을 출력하고, 없으면 "[오류] 등록되지 않은 도구:" 와 name 을 출력하세요 (네 줄)
if func:
    print(func(choice["args"]))
else:
    print("[오류] 등록되지 않은 도구:", name)

### 정답 6-6

In [ ]:
import json
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
# ── 여기까지 ──


def route_tool_call(choice):
    # 1. name 이라는 변수를 만들어 choice 의 tool 을 담으세요
    name = choice["tool"]

    # 2. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요
    func = tool_registry.get(name)

    # 3. func 가 있으면 func 에 choice 의 args 를 넣어 실행한 값을 return 하세요 (두 줄)
    if func:
        return func(choice["args"])

    print("[오류] 등록되지 않은 도구:", name)
    return None


print(route_tool_call({"tool": "count_failed_logins", "args": {"user": "park.js"}}))
print(route_tool_call({"tool": "block_everything", "args": {}}))

### 정답 6-7

In [ ]:
%%writefile tool_router.py
import json
import requests


def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count


def lookup_ip(args):
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]


tool_registry = {}
# 1. tool_registry 에 도구 두 개를 이름과 함수의 짝으로 등록하세요 (두 줄)
tool_registry["count_failed_logins"] = count_failed_logins
tool_registry["lookup_ip"] = lookup_ip


def route_tool_call(choice):
    name = choice["tool"]
    # 2. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요
    func = tool_registry.get(name)

    # 3. func 가 있으면 func 에 choice 의 args 를 넣어 실행한 값을 return 하세요 (두 줄)
    if func:
        return func(choice["args"])

    print("[오류] 등록되지 않은 도구:", name)
    return None

### 정답 6-8

In [ ]:
from llm_client import call_llm, parse_llm_json
from tool_router import route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "park.js 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. text 라는 변수를 만들어 call_llm 에 tools_text + "목표: " + goal 을 넣어 받은 답을 담으세요
text = call_llm(tools_text + "목표: " + goal)

# 2. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요
choice = parse_llm_json(text)

# 3. choice 가 있으면 result 에 route_tool_call(choice) 의 결과를 담고, choice 의 tool, "→", result 를 출력하세요 (세 줄)
if choice:
    result = route_tool_call(choice)
    print(choice["tool"], "→", result)

### 정답 6-9

In [ ]:
from llm_client import call_llm, parse_llm_json
from tool_router import route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "오늘 점심 메뉴를 추천해 주세요."
text = call_llm(tools_text + "목표: " + goal)
# ── 여기까지 ──

# 1. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요
choice = parse_llm_json(text)

# 2. choice 가 있으면 route_tool_call(choice) 의 결과를 출력하고, 없으면 "도구를 고르지 못했습니다" 를 출력하세요 (네 줄)
if choice:
    print(route_tool_call(choice))
else:
    print("도구를 고르지 못했습니다")

### 정답 6-10

In [ ]:
import json
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
choice = {"tool": "count_by_ip", "args": {"ip": "211.45.12.9"}}
# ── 여기까지 ──

# 1. count_by_ip 라는 함수를 만드세요. args 를 받아, normalized_logs.json 에서 ip 가 args 의 ip 와 같은 기록의 수를 return 합니다
# 2. tool_registry 의 "count_by_ip" 칸에 함수 count_by_ip 를 담으세요
# 3. route_tool_call 에 choice 를 넣어 받은 결과를 출력하세요
def count_by_ip(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["ip"] == args["ip"]:
            count = count + 1
    return count


tool_registry["count_by_ip"] = count_by_ip
print(route_tool_call(choice))

### 정답 7-3

In [ ]:
from tool_router import tool_registry, route_tool_call


def lock_account(args):
    # 1. "[조치] 계정 잠금:" 과 args 의 user 를 출력하세요
    print("[조치] 계정 잠금:", args["user"])

    # 2. "locked" 를 return 하세요
    return "locked"


# 3. tool_registry 의 "lock_account" 칸에 함수 lock_account 를 담으세요
tool_registry["lock_account"] = lock_account

print(route_tool_call({"tool": "lock_account", "args": {"user": "guest01"}}))

### 정답 7-4

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
need_approval = ["lock_account"]
choices = [
    {"tool": "lookup_ip", "args": {"ip": "198.51.100.23"}},
    {"tool": "lock_account", "args": {"user": "root"}},
]
# ── 여기까지 ──

for choice in choices:
    # 1. choice 의 tool 이 need_approval 안에 있으면 choice 의 tool 과 "승인 필요" 를 출력하세요 (두 줄)
    if choice["tool"] in need_approval:
        print(choice["tool"], "승인 필요")
    # 2. 없으면 choice 의 tool 과 "바로 실행" 을 출력하세요 (두 줄)
    else:
        print(choice["tool"], "바로 실행")

### 정답 7-5

In [ ]:
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]
choice = {"tool": "lock_account", "args": {"user": "svc_backup"}}
# ── 여기까지 ──

# 1. answer 라는 변수를 만들어 input 으로 "lock_account 을 실행할까요? (y/n) " 를 물어 받은 글자를 담으세요
answer = input("lock_account 을 실행할까요? (y/n) ")

# 2. answer 가 "y" 이면 result 에 route_tool_call(choice) 의 결과를, 아니면 result 에 "held" 를 담으세요 (네 줄)
if answer == "y":
    result = route_tool_call(choice)
else:
    result = "held"

print(result)

### 정답 7-6

In [ ]:
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]
# ── 여기까지 ──


def run_with_gate(choice):
    # 1. choice 의 tool 이 need_approval 안에 있으면 (if 한 줄)
    if choice["tool"] in need_approval:
        # 2. answer 라는 변수를 만들어 input 으로 choice 의 tool + " 을 실행할까요? (y/n) " 를 물어 받은 글자를 담으세요
        answer = input(choice["tool"] + " 을 실행할까요? (y/n) ")
        # 3. answer 가 "y" 가 아니면 "held" 를 return 하세요 (두 줄)
        if answer != "y":
            return "held"

    # 4. route_tool_call(choice) 의 결과를 return 하세요
    return route_tool_call(choice)


print(run_with_gate({"tool": "count_failed_logins", "args": {"user": "lee.yh"}}))
print(run_with_gate({"tool": "lock_account", "args": {"user": "lee.yh"}}))

### 정답 7-7

In [ ]:
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "jung.hw 계정이 몇 번 실패했나"
choice = {"tool": "count_failed_logins", "args": {"user": "jung.hw"}}
records = []
# ── 여기까지 ──

# 1. result 라는 변수를 만들어 route_tool_call 에 choice 를 넣어 받은 값을 담으세요
result = route_tool_call(choice)

# 2. record 라는 변수를 만들어 goal, choice 의 tool, result 를 담은 딕셔너리를 만드세요. 키는 goal, tool, result 입니다
record = {"goal": goal, "tool": choice["tool"], "result": result}

# 3. records 에 record 를 더하세요
records.append(record)

print(records)

### 정답 7-8

In [ ]:
import json

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
records = [
    {"goal": "admin 계정이 몇 번 실패했나", "tool": "count_failed_logins", "result": 4},
    {"goal": "admin 계정을 잠가 달라", "tool": "lock_account", "result": "held"},
]
# ── 여기까지 ──

# 1. "agent_result.json" 을 쓰기로 여세요. 변수 이름은 f 입니다 (with 한 줄)
with open("agent_result.json", "w", encoding="utf-8") as f:
    # 2. records 를 f 에 JSON 으로 저장하세요. 한글 그대로, 두 칸 들여쓰기입니다
    json.dump(records, f, ensure_ascii=False, indent=2)

# 3. 저장한 파일을 다시 읽기로 열어, saved 라는 변수에 읽은 내용을 담으세요 (두 줄)
with open("agent_result.json", encoding="utf-8") as f:
    saved = json.load(f)

print(f"{len(saved)}건 저장했습니다")

### 정답 7-9

In [ ]:
from llm_client import call_llm, parse_llm_json
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]

def run_with_gate(choice):
    if choice["tool"] in need_approval:
        answer = input(choice["tool"] + " 을 실행할까요? (y/n) ")
        if answer != "y":
            return "held"
    return route_tool_call(choice)

tools_text = "쓸 수 있는 도구는 셋입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "3) lock_account: 계정을 잠근다. 인자는 user. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요."
records = []
# ── 여기까지 ──

# 1. text 라는 변수에 call_llm(tools_text + "목표: " + goal) 의 답을, choice 라는 변수에 parse_llm_json(text) 의 결과를 담으세요 (두 줄)
text = call_llm(tools_text + "목표: " + goal)
choice = parse_llm_json(text)

# 2. choice 가 있으면 result 에 run_with_gate(choice) 의 결과를 담고, goal · tool · result 를 담은 딕셔너리를 records 에 더하세요 (세 줄)
if choice:
    result = run_with_gate(choice)
    records.append({"goal": goal, "tool": choice["tool"], "result": result})

print(records)

### 정답 7-10

In [ ]:
import json
from llm_client import call_llm, parse_llm_json
from tool_router import tool_registry, route_tool_call

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
def lock_account(args):
    print("[조치] 계정 잠금:", args["user"])      # 실습이라 출력만 합니다. 실제로 잠그지 않습니다
    return "locked"


tool_registry["lock_account"] = lock_account
need_approval = ["lock_account"]

def run_with_gate(choice):
    if choice["tool"] in need_approval:
        answer = input(choice["tool"] + " 을 실행할까요? (y/n) ")
        if answer != "y":
            return "held"
    return route_tool_call(choice)

tools_text = "쓸 수 있는 도구는 셋입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "3) lock_account: 계정을 잠근다. 인자는 user. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "211.45.12.9 는 어느 나라의 주소인지 알려 주세요.",
    "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요.",
]
records = []
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 text 와 choice 를 만드세요
# 3. choice 가 있으면 run_with_gate 로 실행하고, goal · tool · result 를 records 에 더하세요
# 4. 반복이 끝나면 records 를 "agent_result.json" 으로 저장하세요
for goal in goals:
    text = call_llm(tools_text + "목표: " + goal)
    choice = parse_llm_json(text)
    if choice:
        result = run_with_gate(choice)
        records.append({"goal": goal, "tool": choice["tool"], "result": result})

with open("agent_result.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"{len(records)}건 저장했습니다")